# Spam Filtering with Deep Learning

**Author: Om Shah**


## 1) Environment Setup and File Discovery

This cell loads all the Python libraries the notebook needs - things like pandas for handling data, TensorFlow for building the neural network, and matplotlib for drawing plots.

It also:
- Sets a fixed random seed (`42`) so the model trains the same way every time
- Defines `DEMO_MODE` - set this to `True` before a demo to skip retraining and load the already-saved model instead
- Looks for the required files (`spam.csv`, `train_idx.npy`, `test_idx.npy`, GloVe) in several possible folder locations, so it works regardless of exactly where the files are placed
- Stops immediately with a clear error message if any required file is missing, rather than silently failing later

In [1]:
from pathlib import Path
import io
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from tensorflow.keras import callbacks, layers, models
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DEMO_MODE = True

BASE_DIR = Path(".")
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

spam_candidates = [BASE_DIR / "spam.csv", BASE_DIR / "../spam.csv"]
train_idx_candidates = [
    BASE_DIR / "train_idx.npy",
    BASE_DIR / "outputs/train_idx.npy",
    BASE_DIR / "../train_idx.npy",
]
test_idx_candidates = [
    BASE_DIR / "test_idx.npy",
    BASE_DIR / "outputs/test_idx.npy",
    BASE_DIR / "../test_idx.npy",
]
glove_candidates = [
    BASE_DIR / "glove.twitter.27B.100d.txt",
    BASE_DIR / "glove.twitter.27B.100d (1).txt",
    BASE_DIR / "../glove.twitter.27B.100d.txt",
    BASE_DIR / "../glove.twitter.27B.100d (1).txt",
]


def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


def metrics_row(model_name, y_true, y_pred, score):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_spam": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_spam": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_spam": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_true, score),
        "avg_precision": average_precision_score(y_true, score),
    }


def save_confusion(y_true, y_pred, title, out_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["ham", "spam"])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def save_roc(y_true, score, title, out_path):
    fpr, tpr, _ = roc_curve(y_true, score)
    auc_v = roc_auc_score(y_true, score)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(fpr, tpr, label=f"AUC = {auc_v:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def save_pr(y_true, score, title, out_path):
    precision, recall, _ = precision_recall_curve(y_true, score)
    ap_v = average_precision_score(y_true, score)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(recall, precision, label=f"AP = {ap_v:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title)
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


SPAM_PATH = first_existing(spam_candidates)
TRAIN_IDX_PATH = first_existing(train_idx_candidates)
TEST_IDX_PATH = first_existing(test_idx_candidates)
GLOVE_PATH = first_existing(glove_candidates)

if SPAM_PATH is None:
    raise FileNotFoundError(f"spam.csv not found. Checked: {[str(p.resolve()) for p in spam_candidates]}")
if TRAIN_IDX_PATH is None or TEST_IDX_PATH is None:
    raise FileNotFoundError(
        "Missing split index files. Expected existing train_idx.npy and test_idx.npy. "
        f"Checked train: {[str(p.resolve()) for p in train_idx_candidates]} | "
        f"test: {[str(p.resolve()) for p in test_idx_candidates]}"
    )

print(f"Data file: {SPAM_PATH.resolve()}")
print(f"Train idx file: {TRAIN_IDX_PATH.resolve()}")
print(f"Test idx file: {TEST_IDX_PATH.resolve()}")
print(f"GloVe file: {GLOVE_PATH.resolve() if GLOVE_PATH else 'Not found (will use trainable embeddings)'}")
print(f"Outputs: {OUT_DIR.resolve()}")

Data file: C:\Users\omssh\OneDrive\Desktop\DATAANALYTICS\spam.csv
Train idx file: C:\Users\omssh\OneDrive\Desktop\DATAANALYTICS\train_idx.npy
Test idx file: C:\Users\omssh\OneDrive\Desktop\DATAANALYTICS\test_idx.npy
GloVe file: C:\Users\omssh\OneDrive\Desktop\DATAANALYTICS\glove.twitter.27B.100d (1).txt
Outputs: C:\Users\omssh\OneDrive\Desktop\DATAANALYTICS\outputs


## 2) Load and Validate Dataset

This cell reads `spam.csv` into memory.

- It first tries to open the file using the `cp1252` encoding (common for Windows-saved files). If that fails, it falls back to UTF-8 and skips any characters it cannot read.
- It keeps only the two relevant columns: `v1` (the label - ham or spam) and `v2` (the message text), then renames them to `label_text` and `text`.
- Empty rows are removed.
- The text labels `ham` and `spam` are converted to numbers: `0` for ham, `1` for spam.

At the end it prints how many rows were loaded and what percentage are spam, which acts as a quick sanity check before any training begins.

In [2]:
try:
    raw_df = pd.read_csv(SPAM_PATH, encoding="cp1252")
    used_encoding = "cp1252"
except UnicodeDecodeError:
    with open(SPAM_PATH, "r", encoding="utf-8", errors="ignore") as f:
        raw_df = pd.read_csv(io.StringIO(f.read()))
    used_encoding = "utf-8 (errors='ignore')"

if {"v1", "v2"}.issubset(raw_df.columns):
    data = raw_df[["v1", "v2"]].copy()
else:
    raise ValueError(f"Expected columns v1 and v2. Actual columns: {list(raw_df.columns)}")

data.columns = ["label_text", "text"]
data = data.dropna(subset=["text"]).copy()
data["text"] = data["text"].astype(str)
data = data[data["text"].str.strip() != ""].copy()
data["label"] = data["label_text"].astype(str).str.strip().str.lower().map({"ham": 0, "spam": 1})
data = data.dropna(subset=["label"]).reset_index(drop=True)
data["label"] = data["label"].astype(int)

print(f"Encoding used: {used_encoding}")
print(f"Rows: {len(data)}")
print("Class counts:")
print(data["label"].value_counts().sort_index())
print("Spam rate:", data["label"].mean())

Encoding used: cp1252
Rows: 5572
Class counts:
label
0    4825
1     747
Name: count, dtype: int64
Spam rate: 0.13406317300789664


## 3) Load Fixed Train/Test Split

This cell loads the pre-made split files `train_idx.npy` and `test_idx.npy`.

These files contain the row numbers that belong to the training set and the test set. Because they were created once and saved, the split is identical every time the notebook runs - which means results are consistent and directly comparable with teammates who used the same files.

Two safety checks are applied:
- The training and test sets must not share any rows
- All row numbers must be within the bounds of the loaded dataset

If either check fails, execution stops with an error before any training happens.

In [3]:
train_idx = np.load(TRAIN_IDX_PATH)
test_idx = np.load(TEST_IDX_PATH)

if np.intersect1d(train_idx, test_idx).size != 0:
    raise ValueError("train_idx and test_idx overlap.")
if np.max(train_idx) >= len(data) or np.max(test_idx) >= len(data):
    raise ValueError("Index files refer to rows outside the current dataset length.")

X_all = data["text"].tolist()
y_all = data["label"].values

X_train = [X_all[i] for i in train_idx]
X_test = [X_all[i] for i in test_idx]
y_train = y_all[train_idx]
y_test = y_all[test_idx]

print(f"Train size: {len(train_idx)} | Test size: {len(test_idx)}")
print("Train spam rate:", y_train.mean())
print("Test spam rate:", y_test.mean())

Train size: 4457 | Test size: 1115
Train spam rate: 0.13417096701817366
Test spam rate: 0.1336322869955157


## 4) Text Preprocessing

Neural networks cannot work with raw text - they need numbers. This cell converts every message into a fixed-length sequence of integers.

Steps:
1. **Tokenisation** - each unique word in the training data is assigned a number. Only the 20,000 most common words are kept; anything rarer is treated as unknown.
2. **Sequencing** - every message is converted from words into its list of assigned numbers.
3. **Padding** - messages are different lengths, but the model needs a fixed input size. Every sequence is extended or cut to exactly 80 numbers. Short messages are padded with zeros at the end; long messages are trimmed.

This process is fitted only on the training data so that no information from the test set leaks into the vocabulary.

In [4]:
max_words = 20000
max_len = 80
embedding_dim = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

train_seq = tokenizer.texts_to_sequences(X_train)
test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(train_seq, maxlen=max_len, padding="post", truncating="post")
X_test_pad = pad_sequences(test_seq, maxlen=max_len, padding="post", truncating="post")

word_index = tokenizer.word_index
num_tokens = min(max_words, len(word_index) + 1)

print("Train padded shape:", X_train_pad.shape)
print("Test padded shape:", X_test_pad.shape)
print("Vocab size used:", num_tokens)

Train padded shape: (4457, 80)
Test padded shape: (1115, 80)
Vocab size used: 7920


## 5) Build Embedding Matrix (GloVe)

Each word in the vocabulary needs to be represented as a list of 100 numbers (a vector) that captures its meaning. This cell builds that lookup table.

- If the GloVe file is available, pre-trained vectors from Twitter data are loaded for every word in the vocabulary. GloVe vectors were trained on billions of tweets, so words with similar meanings end up with similar vectors - giving the model a head start.
- If GloVe is not found, small random numbers are used instead. The model can still learn from scratch, just without the head start.

The result is a matrix with one row per vocabulary word. This matrix is passed directly into the model's embedding layer.

In [5]:
embeddings_index = {}
use_glove = GLOVE_PATH is not None

if use_glove:
    needed_words = {w for w, i in word_index.items() if i < max_words}
    with open(GLOVE_PATH, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            if word in needed_words:
                vec = np.asarray(parts[1:], dtype="float32")
                if vec.shape[0] == embedding_dim:
                    embeddings_index[word] = vec

embedding_matrix = np.zeros((num_tokens, embedding_dim), dtype="float32")
if use_glove:
    for word, i in word_index.items():
        if i >= max_words:
            continue
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[i] = vec
else:
    rng = np.random.default_rng(SEED)
    embedding_matrix = rng.normal(0, 0.05, size=(num_tokens, embedding_dim)).astype("float32")
    embedding_matrix[0] = 0.0

print("Using GloVe:", use_glove)
if use_glove:
    print("Matched vectors:", len(embeddings_index))

Using GloVe: True
Matched vectors: 5935


## 6) Build, Train, and Evaluate the Model

This is the main section. It defines the neural network, trains it, and saves all results.

**Model architecture** (`Embedding → Conv1D → GlobalMaxPooling1D → Dense → Dropout → Dense`):
- The **Embedding** layer converts word numbers into their GloVe vectors
- The **Conv1D** layer scans across the message looking for meaningful word patterns (similar to how an n-gram detector works)
- **GlobalMaxPooling1D** picks the strongest signal found across the whole message
- The **Dense** layer learns higher-level combinations of those signals
- **Dropout** randomly switches off 30% of neurons during training to prevent the model from memorising the training data
- The final **Dense** layer outputs a single number between 0 and 1: the probability that the message is spam

**Training**: runs for up to 10 passes through the data. If the validation loss stops improving for 2 consecutive passes, training stops early and the best weights are kept.

**DEMO_MODE**: if set to `True`, the saved model is loaded directly - no training occurs.

**Outputs saved to `outputs/`**:
`dl_model.keras`, `confusion_matrix_dl.png`, `roc_dl.png`, `pr_dl.png`, `training_curves_dl.png`, `test_predictions_dl.csv`

In [6]:
dl_model_path = OUT_DIR / "dl_model.keras"
dl_pred_path = OUT_DIR / "test_predictions_dl.csv"

model = models.Sequential(
    [
        layers.Embedding(
            input_dim=num_tokens,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            trainable=not use_glove,
        ),
        layers.Conv1D(128, 5, activation="relu"),
        layers.GlobalMaxPooling1D(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = None
if DEMO_MODE and dl_model_path.exists():
    model = tf.keras.models.load_model(dl_model_path)
else:
    early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
    history = model.fit(
        X_train_pad,
        y_train,
        validation_split=0.2,
        epochs=10,
        batch_size=64,
        callbacks=[early_stop],
        verbose=1,
    )
    model.save(dl_model_path)

score_dl = model.predict(X_test_pad, verbose=0).ravel()
pred_dl = (score_dl >= 0.5).astype(int)

print("DL classification report:")
print(classification_report(y_test, pred_dl, target_names=["ham", "spam"], digits=4, zero_division=0))

save_confusion(y_test, pred_dl, "Confusion Matrix - Deep Learning", OUT_DIR / "confusion_matrix_dl.png")
save_roc(y_test, score_dl, "ROC - Deep Learning", OUT_DIR / "roc_dl.png")
save_pr(y_test, score_dl, "Precision-Recall - Deep Learning", OUT_DIR / "pr_dl.png")

if history is not None:
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(history.history["loss"], label="train")
    ax[0].plot(history.history["val_loss"], label="val")
    ax[0].set_title("DL Loss")
    ax[0].legend()
    ax[1].plot(history.history["accuracy"], label="train")
    ax[1].plot(history.history["val_accuracy"], label="val")
    ax[1].set_title("DL Accuracy")
    ax[1].legend()
    fig.tight_layout()
    fig.savefig(OUT_DIR / "training_curves_dl.png", dpi=150)
    plt.close(fig)

pd.DataFrame(
    {
        "idx": test_idx,
        "text": X_test,
        "actual": y_test,
        "pred_dl": pred_dl,
        "score_dl": score_dl,
    }
).to_csv(dl_pred_path, index=False)

metrics_rows = []
metrics_rows.append(metrics_row("DeepLearning_CNN", y_test, pred_dl, score_dl))
print(f"Saved DL artifacts to: {OUT_DIR.resolve()}")

DL classification report:
              precision    recall  f1-score   support

         ham     0.9897    0.9959    0.9928       966
        spam     0.9720    0.9329    0.9521       149

    accuracy                         0.9874      1115
   macro avg     0.9809    0.9644    0.9724      1115
weighted avg     0.9873    0.9874    0.9873      1115

Saved DL artifacts to: C:\Users\omssh\OneDrive\Desktop\DATAANALYTICS\outputs


## 7) Metrics Summary

This cell takes the evaluation numbers computed in the previous cell and organises them into a single table:

- **Accuracy** - percentage of all messages classified correctly
- **Precision (spam)** - of all messages the model flagged as spam, how many actually were
- **Recall (spam)** - of all actual spam messages, how many the model caught
- **F1 (spam)** - a single score that balances precision and recall
- **F1 macro** - average F1 across both classes
- **ROC-AUC** - how well the model ranks spam above ham across all possible thresholds
- **Avg Precision** - area under the precision-recall curve

The table is displayed in the notebook and saved to `outputs/metrics_dl.csv`.

In [7]:
metrics_df = pd.DataFrame(metrics_rows)[
    [
        "model",
        "accuracy",
        "precision_spam",
        "recall_spam",
        "f1_spam",
        "f1_macro",
        "roc_auc",
        "avg_precision",
    ]
]

metrics_df.to_csv(OUT_DIR / "metrics_dl.csv", index=False)
display(metrics_df)
print(f"Saved metrics file: {(OUT_DIR / 'metrics_dl.csv').resolve()}")

,model,accuracy,precision_spam,recall_spam,f1_spam,f1_macro,roc_auc,avg_precision
0,DeepLearning_CNN,0.987444,0.972028,0.932886,0.952055,0.972415,0.994209,0.980651


Saved metrics file: C:\Users\omssh\OneDrive\Desktop\DATAANALYTICS\outputs\metrics_dl.csv


## 8) Error Analysis

This cell looks at the messages the model got wrong and prints up to 10 of them.

For each wrong prediction it shows:
- The **actual label** (what it really was)
- The **predicted label** (what the model said)
- The **spam probability score** the model assigned
- The **full message text**

This is useful for the report reflection section - seeing the actual failing examples reveals patterns such as spam written in plain conversational language, or unusual ham messages that look suspicious.

In [8]:
wrong = np.where(pred_dl != y_test)[0]
print("Wrong predictions:", len(wrong))

for i in wrong[:10]:
    actual = "spam" if y_test[i] == 1 else "ham"
    guessed = "spam" if pred_dl[i] == 1 else "ham"
    print("\n---")
    print("Actual:", actual, "| Predicted:", guessed, "| Prob(spam):", float(score_dl[i]))
    print("Text:", X_test[i])

Wrong predictions: 14

---
Actual: spam | Predicted: ham | Prob(spam): 0.06402899324893951
Text: Hi if ur lookin 4 saucy daytime fun wiv busty married woman Am free all next week Chat now 2 sort time 09099726429 JANINExx Callså£1/minMobsmoreLKPOBOX177HP51FL

---
Actual: spam | Predicted: ham | Prob(spam): 0.37810832262039185
Text: ringtoneking 84484

---
Actual: spam | Predicted: ham | Prob(spam): 0.0025435357820242643
Text: Sorry I missed your call let's talk when you have the time. I'm on 07090201529

---
Actual: spam | Predicted: ham | Prob(spam): 0.2249184399843216
Text: Latest News! Police station toilet stolen, cops have nothing to go on!

---
Actual: spam | Predicted: ham | Prob(spam): 0.07626520097255707
Text: Can U get 2 phone NOW? I wanna chat 2 set up meet Call me NOW on 09096102316 U can cum here 2moro Luv JANE xx Callså£1/minmoremobsEMSPOBox45PO139WA

---
Actual: ham | Predicted: spam | Prob(spam): 0.7457234859466553
Text: Hi this is yijue... It's regarding the 3230 textbo

## 9) How to Run Locally (Windows)

First-time setup:
```
python -m venv .venv
.venv\Scripts\activate
pip install tensorflow pandas numpy scikit-learn matplotlib jupyter
jupyter notebook
```

**For the demo (no retraining):**
1. In the first code cell, change `DEMO_MODE = False` to `DEMO_MODE = True`
2. Run **Kernel → Restart & Run All**

The notebook will load the saved model and reproduce all results in seconds.